In [13]:
#%pip install seaborn

In [14]:
import pandas as pd 
import seaborn as sns

df = sns.load_dataset("titanic")
df.head(5)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


In [15]:
print("Checking nulls present in this dataset")
df.isnull().sum()

Checking nulls present in this dataset


survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [16]:
df = df[
    ["survived", "pclass", "sex", "age", "fare", "embarked"]
]


df = df.dropna()
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
Index: 712 entries, 0 to 890
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   survived  712 non-null    int64  
 1   pclass    712 non-null    int64  
 2   sex       712 non-null    str    
 3   age       712 non-null    float64
 4   fare      712 non-null    float64
 5   embarked  712 non-null    str    
dtypes: float64(2), int64(2), str(2)
memory usage: 38.9 KB


survived    0
pclass      0
sex         0
age         0
fare        0
embarked    0
dtype: int64

## Turning Categorical features into numerical

In [17]:
df = pd.get_dummies(
    df, columns = ["sex", "embarked"]
)

df.info()

<class 'pandas.DataFrame'>
Index: 712 entries, 0 to 890
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   survived    712 non-null    int64  
 1   pclass      712 non-null    int64  
 2   age         712 non-null    float64
 3   fare        712 non-null    float64
 4   sex_female  712 non-null    bool   
 5   sex_male    712 non-null    bool   
 6   embarked_C  712 non-null    bool   
 7   embarked_Q  712 non-null    bool   
 8   embarked_S  712 non-null    bool   
dtypes: bool(5), float64(2), int64(2)
memory usage: 31.3 KB


In [18]:
X = df.drop("survived", axis = 1)
y = df["survived"]

print(f"Shape of X is : {X.shape}")
print(f"Shape of y is : {y.shape}")

print(X.head())
print(y.head())

if X.shape[0] == y.shape[0] :
    print("Dataset is good to go . Shapes match , we can continue")


Shape of X is : (712, 8)
Shape of y is : (712,)
   pclass   age     fare  sex_female  sex_male  embarked_C  embarked_Q  \
0       3  22.0   7.2500       False      True       False       False   
1       1  38.0  71.2833        True     False        True       False   
2       3  26.0   7.9250        True     False       False       False   
3       1  35.0  53.1000        True     False       False       False   
4       3  35.0   8.0500       False      True       False       False   

   embarked_S  
0        True  
1       False  
2        True  
3        True  
4        True  
0    0
1    1
2    1
3    1
4    0
Name: survived, dtype: int64
Dataset is good to go . Shapes match , we can continue


## Spliting the dataset and training the decision tree 

In [30]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size = 0.3, random_state = 42   
)

print(X_train.shape)
print(y_train.shape)

model = DecisionTreeClassifier(
    max_depth = 4,
    random_state = 42
)

model.fit(X_train,y_train)
predictions = model.predict(X_val)

print(predictions[:10])

## Checking for overfitting 
train_predictions = model.predict(X_train)

print(f"Train: {accuracy_score(y_train, train_predictions)} ")
print(f"Validation : {accuracy_score(y_val, predictions)}")

(498, 8)
(498,)
[1 1 1 1 0 1 1 0 1 0]
Train: 0.857429718875502 
Validation : 0.6962616822429907


# Daignostics : Handling Overfitting 
### Checking for different depth values so our tree isnt overfitting the training set


In [20]:
for depth in [1, 2, 3, 4, 5, 8, 12]:
    model = DecisionTreeClassifier(
        max_depth = depth,
        random_state = 42
    )
    model.fit(X_train,y_train)
    predictions = model.predict(X_val)
    train_predictions = model.predict(X_train)

    print(f"At depth {depth} : Train = {accuracy_score(train_predictions, y_train)}")
    print(f"At depth {depth} : Validation = {accuracy_score(predictions, y_val)}")

At depth 1 : Train = 0.7873462214411248
At depth 1 : Validation = 0.7482517482517482
At depth 2 : Train = 0.8014059753954306
At depth 2 : Validation = 0.7482517482517482
At depth 3 : Train = 0.8224956063268892
At depth 3 : Validation = 0.7482517482517482
At depth 4 : Train = 0.8347978910369068
At depth 4 : Validation = 0.7132867132867133
At depth 5 : Train = 0.8611599297012302
At depth 5 : Validation = 0.7412587412587412
At depth 8 : Train = 0.9138840070298769
At depth 8 : Validation = 0.7412587412587412
At depth 12 : Train = 0.968365553602812
At depth 12 : Validation = 0.6713286713286714


### Testing the minimum samples required to split : Found that this alone hanldes the overfitting problem too

In [21]:
for samples in [2, 5, 10, 20, 50, 100]:
    model = DecisionTreeClassifier(
        max_depth = None,
        min_samples_split = samples,
        random_state = 42
    )
    model.fit(X_train, y_train)

    print(f"With sample size {samples}: \ntrain = {accuracy_score(y_train, model.predict(X_train))}")
    print(f"validation = {accuracy_score(y_val, model.predict(X_val))}")

With sample size 2: 
train = 0.9859402460456942
validation = 0.6643356643356644
With sample size 5: 
train = 0.9507908611599297
validation = 0.7062937062937062
With sample size 10: 
train = 0.9191564147627417
validation = 0.7272727272727273
With sample size 20: 
train = 0.8892794376098418
validation = 0.7272727272727273
With sample size 50: 
train = 0.8488576449912126
validation = 0.7132867132867133
With sample size 100: 
train = 0.81195079086116
validation = 0.7272727272727273


## Experimented with variable depth and samples and got 75% accuracy which is 2% better than doing varible sample size alone 

In [22]:
for depth in [1, 2, 3, 4, 5, 8, 12]:
    print(f"Statistics at depth {depth}")
    for sampels in [2, 5, 10, 20, 50, 100]:
        model = DecisionTreeClassifier(
            max_depth = depth,
            min_samples_split = sampels,
            random_state = 42
        )
        model.fit(X_train,y_train)
        predictions = model.predict(X_val)
        train_predictions = model.predict(X_train)

        print(f"With sample size {sampels}: \ntrain = {accuracy_score(y_train, model.predict(X_train))}")
        print(f"validation = {accuracy_score(y_val, model.predict(X_val))}")

Statistics at depth 1
With sample size 2: 
train = 0.7873462214411248
validation = 0.7482517482517482
With sample size 5: 
train = 0.7873462214411248
validation = 0.7482517482517482
With sample size 10: 
train = 0.7873462214411248
validation = 0.7482517482517482
With sample size 20: 
train = 0.7873462214411248
validation = 0.7482517482517482
With sample size 50: 
train = 0.7873462214411248
validation = 0.7482517482517482
With sample size 100: 
train = 0.7873462214411248
validation = 0.7482517482517482
Statistics at depth 2
With sample size 2: 
train = 0.8014059753954306
validation = 0.7482517482517482
With sample size 5: 
train = 0.8014059753954306
validation = 0.7482517482517482
With sample size 10: 
train = 0.8014059753954306
validation = 0.7482517482517482
With sample size 20: 
train = 0.8014059753954306
validation = 0.7482517482517482
With sample size 50: 
train = 0.8014059753954306
validation = 0.7482517482517482
With sample size 100: 
train = 0.8014059753954306
validation = 0.748

# Random Forests

In [23]:
from sklearn.ensemble import RandomForestClassifier

for estimaters in [20, 50, 100]:
    model = RandomForestClassifier(
        n_estimators = estimaters,
        max_depth = 4,
        min_samples_split = 50,
        random_state = 42
    )

    model.fit(X_train, y_train)

    print(f"With {estimaters} trees:")
    print(f"Accuracy of random forest : \nTrain: {accuracy_score(model.predict(X_train),y_train)}")
    print(f"Validation: {accuracy_score(model.predict(X_val),y_val)}")

With 20 trees:
Accuracy of random forest : 
Train: 0.8260105448154658
Validation: 0.7832167832167832
With 50 trees:
Accuracy of random forest : 
Train: 0.827768014059754
Validation: 0.7692307692307693
With 100 trees:
Accuracy of random forest : 
Train: 0.8224956063268892
Validation: 0.7762237762237763


One important difference from your single Decision Tree:

Random Forest doesn't necessarily get closer to 100% training accuracy as depth increases, because it introduces randomness when building the trees.

In [24]:
for depth in [2, 4, 8, 12, None]:
    model = RandomForestClassifier(
        n_estimators = 100,
        max_depth = depth,
        min_samples_split = 50,
        random_state = 42
    )

    model.fit(X_train, y_train)

    print(f"At {depth} depth:")
    print(f"Accuracy of random forest : \nTrain: {accuracy_score(model.predict(X_train),y_train)}")
    print(f"Validation: {accuracy_score(model.predict(X_val),y_val)}")

At 2 depth:
Accuracy of random forest : 
Train: 0.7926186291739895
Validation: 0.7552447552447552
At 4 depth:
Accuracy of random forest : 
Train: 0.8224956063268892
Validation: 0.7762237762237763
At 8 depth:
Accuracy of random forest : 
Train: 0.8400702987697716
Validation: 0.7902097902097902
At 12 depth:
Accuracy of random forest : 
Train: 0.8418277680140598
Validation: 0.7692307692307693
At None depth:
Accuracy of random forest : 
Train: 0.8383128295254832
Validation: 0.7762237762237763


# XGBoost 


In [39]:
from xgboost import XGBClassifier

for estimaters in [20, 50, 100, 200, 500]:
    print(f"With {estimaters} estimaters:")
    
    for rate in [0.01, 0.03, 0.5, 0.1, 0.2, 0.3]:
        print(f"\nWith learning rate = {rate}")

        model = XGBClassifier(
            n_estimators = estimaters,
            learning_rate = rate,
            max_depth = 4,
            random_state = 42
        )

        model.fit(X_train,y_train)
        print(f"Accuracy \ntrain: {accuracy_score(y_train, model.predict(X_train))} \nValidation: {accuracy_score(y_val, model.predict(X_val))} ")
        print(model.get_booster().num_boosted_rounds())

With 20 estimaters:

With learning rate = 0.01
Accuracy 
train: 0.606425702811245 
Validation: 0.5700934579439252 
20

With learning rate = 0.03
Accuracy 
train: 0.8473895582329317 
Validation: 0.7476635514018691 
20

With learning rate = 0.5
Accuracy 
train: 0.9196787148594378 
Validation: 0.7850467289719626 
20

With learning rate = 0.1
Accuracy 
train: 0.8835341365461847 
Validation: 0.7336448598130841 
20

With learning rate = 0.2
Accuracy 
train: 0.8975903614457831 
Validation: 0.7476635514018691 
20

With learning rate = 0.3
Accuracy 
train: 0.9156626506024096 
Validation: 0.7897196261682243 
20
With 50 estimaters:

With learning rate = 0.01
Accuracy 
train: 0.8413654618473896 
Validation: 0.7523364485981309 
50

With learning rate = 0.03
Accuracy 
train: 0.8734939759036144 
Validation: 0.7242990654205608 
50

With learning rate = 0.5
Accuracy 
train: 0.9578313253012049 
Validation: 0.7570093457943925 
50

With learning rate = 0.1
Accuracy 
train: 0.9116465863453815 
Validation: 

In [53]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=4,
    early_stopping_rounds=20,
    eval_metric="error",
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("Best iteration:", model.best_iteration)
print("Best validation error:", model.best_score)

train_acc = accuracy_score(y_train, model.predict(X_train))
val_acc = accuracy_score(y_val, model.predict(X_val))

print("Train accuracy:", train_acc)
print("Validation accuracy:", val_acc)

Best iteration: 2
Best validation error: 0.2336448598130841
Train accuracy: 0.8232931726907631
Validation accuracy: 0.7663551401869159
